# 02 - Compare the PDF with a 300 dpi TIFF

Render every page of the included local PDF to a grayscale, lossless multi-page TIFF, then send both representations to the same Content Understanding analyzer. This is a diagnostic experiment for faxed/scanned input, not an automatic conversion step for born-digital PDFs.

Rendering changes the input: the submitted TIFF contains pixels, not the PDF's text layer. Increasing the raster resolution does not recover detail absent from the source or guarantee that a thin sign survives OCR.

This notebook runs independently. Follow [setup instructions](../README.md), supply an approved short test document, and run from this directory. The TIFF, results and any document text remain in the ignored `output/` directory.

In [ ]:
import hashlib, json, os
from pathlib import Path

import pandas as pd
import pypdfium2 as pdfium
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.identity import DefaultAzureCredential
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
PDF = Path('../input/statement.pdf')
DPI = 300
ANALYZER_ID = 'prebuilt-layout'
API_VERSION = '2025-11-01'
KEYWORD = 'property tax'
OUT = Path('../output/02-render')
OUT.mkdir(parents=True, exist_ok=True)
print('Input:', PDF, '| bytes:', PDF.stat().st_size)

## Render a lossless TIFF - local only
Adapted from the existing [rasterize_pdf_to_tiff](https://github.com/jomedinagomez/azure-ai-foundry-insurance/blob/f6949572c5aaa6bc77b1ac592a92affc62d98346/demo/sov/preprocess/pdf_render.py) implementation. Reused: PDFium's `dpi / 72` rendering scale and Pillow's `save_all`, `append_images` and LZW TIFF encoding. Changes: 300 dpi grayscale rather than 800 dpi RGB, with explicit resource cleanup and no cropping. See [PDFium](https://pypdfium2.readthedocs.io/en/stable/python_api.html) and [Pillow TIFF](https://pillow.readthedocs.io/en/stable/handbook/image-file-formats.html#tiff).

Use a short diagnostic file: this simple example holds the rendered pages in memory. All source pages are preserved, and the original PDF is not modified.

In [ ]:
TIFF = OUT / f'{PDF.stem}_{DPI}dpi.tiff'
frames = []
with pdfium.PdfDocument(str(PDF)) as document:
    for page in document:
        bitmap = page.render(scale=DPI / 72)
        try:
            frames.append(bitmap.to_pil().convert('L'))
        finally:
            bitmap.close()
if not frames:
    raise ValueError('The PDF has no pages to render.')
try:
    frames[0].save(TIFF, format='TIFF', save_all=True, append_images=frames[1:],
                   compression='tiff_lzw', dpi=(DPI, DPI))
    print('Saved', len(frames), 'pages:', TIFF, '| bytes:', TIFF.stat().st_size)
finally:
    for frame in frames:
        frame.close()

## Analyze both representations - Azure calls
The SDK request flow below is adapted from Microsoft's [binary-analysis sample](https://github.com/Azure/azure-sdk-for-python/blob/53023a2cd9adf255cff520a99985fff84dc58d7d/sdk/contentunderstanding/azure-ai-contentunderstanding/samples/sample_analyze_binary.py). Microsoft copyright and MIT terms: [LICENSE](../LICENSE). The same analyzer ID and GA API version are used for both files. No completion model is required by the default `prebuilt-layout` analyzer.

The next setup cell creates the client. The following comparison cell makes two new, billable requests every time it runs. It does not create analyzers or update deployments.

In [ ]:
endpoint = os.getenv('CONTENTUNDERSTANDING_ENDPOINT') or os.environ['APP_CONTENT_UNDERSTANDING_ENDPOINT']
client = ContentUnderstandingClient(
    endpoint=endpoint,
    credential=DefaultAzureCredential(process_timeout=60),
    api_version=API_VERSION,
)

def analyze_file(path, label):
    file_bytes = path.read_bytes()
    poller = client.begin_analyze_binary(analyzer_id=ANALYZER_ID, binary_input=file_bytes)
    result = poller.result()
    markdown = '\n\n'.join(content.markdown or '' for content in result.contents)
    (OUT / f'{label}.json').write_text(json.dumps(result.as_dict(), indent=2), encoding='utf-8')
    (OUT / f'{label}.md').write_text(markdown, encoding='utf-8')
    evidence = {'case': label, 'file': path.name, 'sha256': hashlib.sha256(file_bytes).hexdigest(),
                'analyzer': ANALYZER_ID, 'api_version': API_VERSION}
    (OUT / f'{label}.input.json').write_text(json.dumps(evidence, indent=2), encoding='utf-8')
    return markdown

In [ ]:
inputs = {'original_pdf': PDF, 'rendered_tiff': TIFF}
texts = {}
for label, path in inputs.items():
    print('Analyzing', label, 'with', ANALYZER_ID)
    texts[label] = analyze_file(path, label)

## Inspect the sign and amount in context
Compare the affected row with the original pixels, not just whether the output contains a minus somewhere. Inspect the saved full response for available word/line grounding and confidence. If the label is not found, review the complete Markdown manually rather than treating that as a passing case.

Keep both results, input hashes and the exact source document for an approved reproduction. A successful rendered variant is evidence of an input-path difference, not proof of a universal 300 dpi fix. Re-test the application's analyzer and non-failing documents before changing an ingestion pipeline.

In [ ]:
for label, text in texts.items():
    print('\n---', label, '---')
    lines = text.splitlines()
    matches = [i for i, line in enumerate(lines) if KEYWORD.casefold() in line.casefold()]
    if not matches:
        print('Label not found. Inspect the complete saved Markdown; no accuracy conclusion was made.')
    for i in matches[:3]:
        print('\n'.join(lines[max(0, i - 1):i + 3]))